# **Netflix Prize Data Analysis**

## **Gerekli Paket ve Verilerin Yüklenmesi**

In [1]:
import numpy as np
import pandas as pd

print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")

numpy version: 1.23.5
pandas version: 2.0.3


In [ ]:
!pip install numpy==1.23.5 pandas==2.0.3 mxnet==1.9.1 d2l==1.0.3

In [ ]:
import os

os.kill(os.getpid(), 9)

In [3]:
import numpy as np
import pandas as pd
import mxnet as mx
import d2l

print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"mxnet version: {mx.__version__}")
print(f"d2l version: {d2l.__version__}")

numpy version: 1.23.5
pandas version: 2.0.3
mxnet version: 1.9.1
d2l version: 1.0.3


In [4]:
!wget https://raw.githubusercontent.com/yunuscvlk/netflix-prize-data-analysis/refs/heads/main/testdata/random100ksample.csv

--2024-12-18 13:54:15--  https://raw.githubusercontent.com/yunuscvlk/netflix-prize-data-analysis/refs/heads/main/testdata/random100ksample.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1598226 (1.5M) [text/plain]
Saving to: ‘random100ksample.csv’

random100ksample.cs 100%[===================>]   1.52M  --.-KB/s    in 0.01s   

2024-12-18 13:54:16 (147 MB/s) - ‘random100ksample.csv’ saved [1598226/1598226]



## **Veri Yükleme ve Hazırlık Fonksiyonları**

Bu fonksiyonlar, film öneri modelinin eğitilmesi için gereken veri hazırlık aşamalarını yönetir. `read_data` fonksiyonu veriyi okur ve temel istatistikleri çıkarırken, `split_data` veri setini eğitim ve test setlerine böler. `load_data` veriyi uygun formata dönüştürür ve `split_and_load` fonksiyonu tüm bu işlemleri birleştirerek, veri setlerini eğitim için hazır hale getirir. Bu adımlar, modelin doğru şekilde eğitilmesi ve değerlendirilmesi için kritik öneme sahiptir.

In [5]:
import pandas as pd

from d2l import mxnet as d2l  # d2l (Dive into Deep Learning), eğitim amacıyla kullanılan bir kütüphanedir.
from mxnet import gluon, np, npx  # mxnet, derin öğrenme için kullanılan bir framework'tür. gluon veri yükleme, np ve npx matematiksel işlemler için kullanılır.

npx.set_np()  # NumPy benzeri bir davranışı MXNet'te etkinleştirir.

def read_data(filename):
    """
    Bir CSV dosyasını okur ve veri hakkında temel istatistikleri çıkarır.

    Args:
        filename (str): Okunacak CSV dosyasının yolu.

    Returns:
        tuple:
            - data (pd.DataFrame): Dosyadan okunan veri.
            - num_customers (int): Verideki unique customer sayısı.
            - num_movies (int): Verideki unique movie sayısı.
    """

    data = pd.read_csv(filename, sep=';')

    num_customers = data["CustomerID"].unique().shape[0]
    num_movies = data["MovieID"].unique().shape[0]

    return data, num_customers, num_movies

def split_data(data, test_ratio=0.2):
    """
    Veriyi verilen test oranına göre eğitim ve test array'lere ayırır.

    Args:
        data (pd.DataFrame): Ayrılacak veri.
        test_ratio (float): Test array'i olarak kullanılacak veri oranı (varsayılan 0.2).

    Returns:
        tuple:
            - train_data (pd.DataFrame): Eğitim DataFrame'i.
            - test_data (pd.DataFrame): Test DataFrame'i.
    """

    # Veriyi eğitim ve test seti olarak bölmek için kullanılır.
    # Rastgele bir mask (True/False array) oluşturulur. %80'i eğitim, %20'si test için ayrılır.
    mask = [True if x == 1 else False for x in np.random.uniform(0, 1, (len(data))) < 1 - test_ratio]

    # Eğitim verisinin mask'inin ters çevrilerek test verisi mask'inin oluşturulması.
    neg_mask = [not x for x in mask]

    # Mask'lere göre eğitim ve test veri setleri ayrılır.
    train_data, test_data = data[mask], data[neg_mask]

    return train_data, test_data

def load_data(data, num_customers, num_movies):
    """
    Veri setinden customer, movie ve rate bilgilerini çıkarır ve bunları NumPy array'lerine dönüştürür.

    Args:
        data (pd.DataFrame): İşlenecek veri seti.
        num_customers (int): Unique customer sayısı.
        num_movies (int): Unique movie sayısı.

    Returns:
        tuple:
            - customers (np.ndarray): Customer array'i (0 tabanlı).
            - movies (np.ndarray): Movie array'i (0 tabanlı).
            - rates (np.ndarray): Rate array'i.
    """

    customers, movies, rates = [], [], []

    for line in data.itertuples():
        # Customer ve movie için array'i 0 tabanlı yapar, rate bilgisi alınır.
        customer_index, movie_index, rate = int(line[1] - 1), int(line[2] - 1), int(line[3])

        customers.append(customer_index)
        movies.append(movie_index)
        rates.append(rate)

    return np.array(customers), np.array(movies), np.array(rates)

def split_and_load(filename, test_ratio=0.2, batch_size=256):
    """
    Veri setini okur, eğitim ve test array'lerine böler ve her iki array için DataLoader hazırlar.

    Args:
        filename (str): Okunacak CSV dosyasının yolu.
        test_ratio (float): Test array'i olarak kullanılacak veri oranı (varsayılan 0.2).
        batch_size (int): Her bir verinin batch boyutu (varsayılan 256).

    Returns:
        tuple:
            - num_customers (int): Veri array'indeki unique customer sayısı.
            - num_movies (int): Veri array'indeki unique movie sayısı.
            - train_iter (DataLoader): Eğitim array'i için DataLoader.
            - test_iter (DataLoader): Test array'i için DataLoader.
    """

    data, num_customers, num_movies = read_data(filename) # Veriyi okur ve temel istatistikleri alır.

    train_data, test_data = split_data(data, test_ratio) # Veriyi eğitim ve test olarak böler.

    # Eğitim ve test veri array'lerini yükler.
    train_customers, train_movies, train_rates = load_data(train_data, num_customers, num_movies)
    test_customers, test_movies, test_rates = load_data(test_data, num_customers, num_movies)

    # Eğitim ve test verilerini ArrayDataset olarak oluşturur.
    train_set = gluon.data.ArrayDataset(train_customers, train_movies, train_rates)
    test_set = gluon.data.ArrayDataset(test_customers, test_movies, test_rates)

    # Veri setlerini DataLoader ile batch halinde iteratör olarak hazırlar.
    train_iter = gluon.data.DataLoader(train_set, shuffle=True, batch_size=batch_size) # Eğitim verisini rastgele karıştırır.
    test_iter = gluon.data.DataLoader(test_set, batch_size=batch_size) # Test verisini batch halinde iteratör yapar.

    return num_customers, num_movies, train_iter, test_iter

filename, test_ratio, batch_size = 'random100ksample.csv', 0.2, 256
num_customers, num_movies, train_iter, test_iter = split_and_load(filename, test_ratio, batch_size)

## **Matrix Factorization Modeli**

Bu sınıf, öneri sistemleri için Matrix Factorization modelini tanımlar. Model, her kullanıcı ve film için gizli faktörler ve bias terimleri öğrenerek kullanıcıların filme vereceği puanları tahmin eder. Bu yaklaşım, kullanıcıların geçmişteki tercihleri ve filmlerin özelliklerine dayalı olarak daha kişiselleştirilmiş öneriler yapılmasını sağlar.


In [6]:
from mxnet.gluon import nn  # MXNet kütüphanesinin Gluon modülünden 'nn' sınıfını içeri aktarır, bu sınıf ağ yapıları oluşturmak için kullanılır.

class MatrixFactorization(nn.Block):
    """
    Öneri sistemleri için Matrix Factorization modelini MXNet Gluon framework'ü kullanarak tanımlar.
    Bu model, customer'lar ve movie'ler için gizli faktörleri öğrenerek puan tahminleri veya etkileşimleri sağlar.
    """

    def __init__(self, num_customers, num_movies, num_factors, **kwargs):
        """
        MatrixFactorization modelini başlatır, customer ve movie embedding'lerini ve bias terimlerini oluşturur.

        Args:
            num_customers (int): Verisetindeki benzersiz customer sayısı.
            num_movies (int): Verisetindeki benzersiz movie sayısı.
            num_factors (int): Embedding'ler için gizli faktör sayısı.
            **kwargs: Üst sınıf (nn.Block) için ek anahtar kelime argümanları.

        Attributes:
            customer_embedding (nn.Embedding): Customer'lar için embedding katmanı.
            movie_embedding (nn.Embedding): Movie'ler için embedding katmanı.
            customer_bias (nn.Embedding): Customer'lar için bias (sapma) terimleri katmanı.
            movie_bias (nn.Embedding): Movie'ler için bias (sapma) terimleri katmanı.
        """

        super(MatrixFactorization, self).__init__(**kwargs)

        # Customer'ların gizli faktör temsillerini (embedding) öğrenmek için embedding katmanı.
        self.customer_embedding = nn.Embedding(num_customers, num_factors)

        # Movie'lerin gizli faktör temsillerini (embedding) öğrenmek için embedding katmanı.
        self.movie_embedding = nn.Embedding(num_movies, num_factors)

        # Customer'lar için bias (sapma) terimleri katmanı.
        self.customer_bias = nn.Embedding(num_customers, 1)

        # Movie'ler için bias (sapma) terimleri katmanı.
        self.movie_bias = nn.Embedding(num_movies, 1)

    def forward(self, customers, movies):
        """
        Verilen CustomerID ve MovieID'leri için tahmin edilen puanları hesaplar.

        Args:
            customers (ndarray): CustomerID'lerini içeren array.
            movies (ndarray): MovieID'lerini içeren array.

        Returns:
            ndarray: Customer'lar ve movie'ler arasındaki tahmin edilen puanlar.
        """

        # Customer'lar için embedding'leri çıkarır.
        customer_vecs = self.customer_embedding(customers)

        # Movie'ler için embedding'leri çıkarır.
        movie_vecs = self.movie_embedding(movies)

        # Customer ve movie embedding'lerinin noktasal çarpımını alır.
        # Daha sonra her satırın toplamını hesaplar ve bias (sapma) terimlerini ekler.
        preds = (
              (customer_vecs * movie_vecs).sum(axis=1) +  # Customer ve movie embedding'lerinin noktasal çarpımını alır ve her satırın toplamını hesaplar.
              self.customer_bias(customers).squeeze() +  # Customer'ların sapmalarını ekler. 'squeeze' ile boyutları sıkıştırır.
              self.movie_bias(movies).squeeze()  # Movie'lerin sapmalarını ekler. 'squeeze' ile boyutları sıkıştırır.
            )

        return preds

## **Film Önerisi Fonksiyonu**

Bu fonksiyon, film önerisi sistemi için bir modelin eğitimini gerçekleştirir. Modelin performansını izleyerek doğruluğu artırmak amacıyla her epoch sonunda RMSE değerini hesaplar. Modelin parametreleri, Adam optimizer ve L2Loss kayıp fonksiyonu ile güncellenir ve bu süreç boyunca modelin doğruluğu sürekli olarak izlenir. Bu, öneri sisteminin kişiselleştirilmiş ve doğru sonuçlar üretmesine yardımcı olur.


In [7]:
from mxnet import autograd, nd, init  # autograd: geri yayılım için türev hesaplamayı sağlar, nd: yüksek performanslı diziler (ndarray), init: model parametrelerini başlatmak için kullanılır.
from mxnet.gluon import Trainer, loss as gloss  # Trainer: model parametrelerini optimize etmek için kullanılır, loss: kayıp fonksiyonlarını sağlar (örneğin, L2Loss).

def train_recommender(net, train_iter, test_iter, num_epochs, lr, wd, ctx):
    """
    Verilen eğitim verisiyle bir öneri sistemi modelini eğitir.

    Args:
        net (gluon.Block): Eğitilecek olan sinir ağı modeli.
        train_iter (iterable): Eğitim verisi sağlayan veri iteratörü.
        test_iter (iterable): Test verisi sağlayan veri iteratörü.
        num_epochs (int): Eğitim için kullanılacak epoch sayısı.
        lr (float): Optimizer için öğrenme oranı.
        wd (float): Optimizer için ağırlık çürütme (L2 regularizasyonu).
        ctx (mxnet.context.Context): Eğitim için kullanılacak cihaz (CPU veya GPU).

    Returns:
        None: Bu fonksiyon her epoch sonunda eğitim RMSE değerini yazdırır.

    Notes:
        - Modelin parametreleri normal dağılımla (std=0.01) başlatılır, bu sayede ağırlıklar küçük rastgele değerler alarak daha hızlı ve etkili bir şekilde öğrenme sağlanır.
        - Eğitim için Adam optimizasyon algoritması kullanılır.
        - L2Loss kayıp fonksiyonu kullanılır.
        - Eğitim belirtilen epoch sayısı kadar yapılır.
        - Her epoch sonunda RMSE (Kök Ortalama Kare Hata) hesaplanır ve yazdırılır.
    """

    # Modeli normal bir dağılımla başlat (standart sapma = 0.01)
    net.initialize(ctx=ctx, force_reinit=True, init=init.Normal(0.01))

    # Adam optimizasyon algoritması kullanılarak modelin parametrelerini eğitmek için bir trainer oluştur.
    trainer = Trainer(net.collect_params(), 'adam', {'learning_rate': lr, 'wd': wd})

    # L2Loss, tahmin ve gerçek değerler arasındaki kareler farkının ortalamasını alır.
    loss = gloss.L2Loss()

    for epoch in range(num_epochs):
        # Birikimli istatistikler için bir sayaç (toplam kayıp, örnek sayısı, batch sayısı)
        metric = d2l.Accumulator(3)

        for i, (customers, movies, rates) in enumerate(train_iter):
            # Verileri eğitim için belirtilen cihaza (CPU/GPU) taşır.
            customers, movies, rates = customers.as_in_ctx(ctx), movies.as_in_ctx(ctx), rates.as_in_ctx(ctx)

            with autograd.record():
                # Modelin tahminlerini hesapla
                preds = net(customers, movies)

                # Tahminler ile gerçek değerler arasındaki kaybı hesapla
                l = loss(preds, rates)

            # Kaybın türevlerini hesapla (geri yayılım)
            l.backward()

            # Optimizer ile modelin ağırlıklarını güncelle
            trainer.step(batch_size=rates.shape[0])

            # Toplam kayıp, veri örneklerinin sayısı ve batch sayısını güncelle
            metric.add(l.sum().item(), rates.size, 1)

        # RMSE'yi (Kök Ortalama Kare Hata) hesapla
        train_rmse = nd.sqrt(nd.array([metric[0]]) / nd.array([metric[1]]))

        print(f'epoch {epoch + 1}, train RMSE: {train_rmse.asscalar():.4f}')

## **Matrix Factorization Modeli ile Film Öneri Sistemi Eğitimi ve İyileştirmesi**

### **Parametreler ve Seçim Kriterleri**

#### **``num_factors = 20``**

- **Açıklama**: Bu, embedding boyutunu belirtir. Her customer ve movie için oluşturulacak gizli faktörlerin sayısını temsil eder. Bu değer, customer'lar ve movie'ler arasındaki ilişkilerin modellenmesi için kullanılan gizli katmanların sayısıdır.
- **Nasıl Belirlenir?**: Bu değer, genellikle deneme-yanılma yöntemi ile ya da teorik bir anlayışa dayalı olarak seçilir. Çok küçük bir değer, modelin yetersiz öğrenmesine yol açabilirken, çok büyük bir değer modelin aşırı öğrenmesine (overfitting) neden olabilir. Genellikle 10-50 arasında bir değer yaygın olarak kullanılır.

#### **``num_epochs = 20``**

- **Açıklama**: Modelin eğitiminde kaç kez verilerin üzerinden geçileceğini belirler. Bir epoch, modelin tüm eğitim verisini bir kez işlemesi anlamına gelir.
- **Nasıl Belirlenir?**: Epoch sayısı, modelin doğruluğunu artırmak için yeterli eğitim süresi sağlamak amacıyla seçilir. Daha fazla epoch, modelin daha fazla öğrenmesini sağlar ancak aşırı öğrenmeye (overfitting) yol açabilir. Genellikle 10-50 arasında bir değer seçilir, ancak modelin öğrenme hızına göre ayarlanabilir.

#### **``lr = 0.005``**

- **Açıklama**: Öğrenme oranı, optimizasyon algoritmasının her adımda modelin parametrelerini ne kadar değiştireceğini belirler.
- **Nasıl Belirlenir?**: Öğrenme oranı, genellikle küçük bir değerle başlanır ve modelin doğruluğu izlenerek yavaşça artırılabilir. Eğer öğrenme oranı çok küçükse, model çok yavaş öğrenir. Eğer çok büyükse, model yanlış yönde öğrenebilir. Genellikle 0.001 ile 0.1 arasında bir değer kullanılır, ancak bu değer farklı model yapılarına göre değişebilir.

#### **``wd = 0``**

- **Açıklama**: Ağırlık çürütme (L2 düzenlileme), modelin ağırlıklarını küçük tutmaya çalışarak aşırı öğrenmeyi (overfitting) engellemeye yardımcı olur. `wd` parametresi, ağırlık çürütme oranını belirler.
- **Nasıl Belirlenir?**: Bu değer, modelin overfitting durumunu önlemek için kullanılır. Eğer model çok karmaşıksa veya eğitim verisi sınırlıysa, ağırlık çürütme parametresi artırılabilir. Genellikle 0 ile 0.1 arasında bir değer kullanılır, ancak bu değerin de modelin özelliklerine göre değiştirilmesi gerekebilir.

#### **``batch_size = 256``**

- **Açıklama**: Batch size, modelin eğitim sırasında her adımda kaç örnek üzerinde işlem yapacağını belirler. Bu, her güncelleme adımında kullanılan veri örneklerinin sayısını ifade eder.
- **Nasıl Belirlenir?**: Batch size, modelin eğitim hızını ve performansını etkileyen önemli bir parametredir. Küçük bir batch size, daha doğru ve hızlı parametre güncellemeleri sağlayabilir ancak daha fazla işlem süresi gerektirir. Büyük bir batch size, daha hızlı eğitim sağlar ancak her adımda daha az güncelleme yapılır. Genellikle 32, 64, 128, 256 gibi değerler kullanılır.


In [8]:
# GPU'yu kullanmak için d2l kütüphanesinden try_gpu fonksiyonunu çağırıyoruz.
# Bu, modelin GPU üzerinde çalışıp çalışmayacağını belirler. Eğer GPU mevcutsa, GPU'yu kullanacaktır.
# Eğer GPU kullanılabilir değilse, CPU kullanılacaktır.
ctx = d2l.try_gpu()

# Embedding boyutunu belirliyoruz, yani her bir customer ve movie için temsil edilen gizli faktör sayısı.
# Burada, 20 faktör kullanılacaktır. Bu, her bir customer'ın ve movie'nin gizli temsili için 20 boyutlu bir vektör kullanılacağı anlamına gelir.
num_factors = 20

# MatrixFactorization sınıfını kullanarak bir öneri sistemi modeli oluşturuyoruz.
# Bu model, customer'lar ve movie'ler arasındaki etkileşimleri öğrenmek için kullanılacaktır.
# num_customers: Customer sayısı, num_movies: Movie sayısı, num_factors: Her customer ve movie için temsil edilen gizli faktör sayısı.
net = MatrixFactorization(num_customers, num_movies, num_factors)

# Eğitim parametrelerini belirliyoruz:
# num_epochs: Modelin eğitiminde kaç kez verilerin üzerinden geçileceğini belirler (20 epoch).
# lr: Öğrenme oranı, modelin her adımda ne kadar güncelleneceğini belirler (0.005).
# wd: Ağırlık düşüşü (L2 düzenlileme), modelin aşırı öğrenmesini engellemek için kullanılır (0, burada düzenleme yapılmıyor).
# batch_size: Her bir adımda işlenecek örnek sayısı (256).
num_epochs, lr, wd, batch_size = 20, 0.005, 0, 256

# Öneri sistemini eğitmek için train_recommender fonksiyonunu çağırıyoruz.
# train_recommender fonksiyonu:
# - net: Eğitimde kullanılacak model,
# - train_iter: Eğitim verisi iterator'ı,
# - test_iter: Test verisi iterator'ı,
# - num_epochs: Eğitim süresi,
# - lr: Öğrenme oranı,
# - wd: Ağırlık düşüşü (L2 düzenlileme),
# - ctx: GPU veya CPU kullanılacak ortam.
train_recommender(net, train_iter, test_iter, num_epochs, lr, wd, ctx)

epoch 1, train RMSE: 1.4799
epoch 2, train RMSE: 0.8125
epoch 3, train RMSE: 0.7571
epoch 4, train RMSE: 0.7395
epoch 5, train RMSE: 0.7284
epoch 6, train RMSE: 0.7208
epoch 7, train RMSE: 0.7143
epoch 8, train RMSE: 0.7094
epoch 9, train RMSE: 0.7054
epoch 10, train RMSE: 0.7020
epoch 11, train RMSE: 0.6990
epoch 12, train RMSE: 0.6957
epoch 13, train RMSE: 0.6937
epoch 14, train RMSE: 0.6914
epoch 15, train RMSE: 0.6897
epoch 16, train RMSE: 0.6881
epoch 17, train RMSE: 0.6871
epoch 18, train RMSE: 0.6860
epoch 19, train RMSE: 0.6853
epoch 20, train RMSE: 0.6844


## **RMSE Hesaplama Fonksiyonu**

Bu fonksiyon, modelin başarımını değerlendirmek için kullanılır. Test verisi üzerinden modelin tahminlerini karşılaştırarak **RMSE** değerini hesaplar. Daha düşük RMSE, modelin doğruluğunun yüksek olduğunu ve gerçek dünya verilerine ne kadar iyi uyum sağladığını gösterir.

In [9]:
from mxnet import nd # mxnet kütüphanesinden nd modülünü içe aktarıyoruz, bu modül NDArray işlemleri için kullanılır.

def evaluate_rmse(net, data_iter, ctx):
    """
    Öneri modelinin Kök Ortalama Kare Hata (RMSE) değerini hesaplar.

    Bu fonksiyon, tahmin edilen puanlar ile gerçek puanlar arasındaki RMSE'yi hesaplar
    ve verilen veri kümesi üzerinde tüm test verisi için bu değeri hesaplar.

    Args:
        net (gluon.Block): Tahminler yapmak için kullanılan eğitimli öneri modeli.
        data_iter (mxnet.gluon.data.DataLoader): CustomerID'leri, MovieID'leri ve gerçek puanları içeren verilerin döngüleyicisi.
        ctx (mxnet.Context): Hesaplamaların yapılacağı cihaz bağlamı (CPU veya GPU).

    Returns:
        float: Verilen veri için hesaplanan RMSE (Kök Ortalama Kare Hata) değeri.

    Notes:
        - Model, test verileri üzerinden tahminler yaparak hata miktarını ölçer.
        - RMSE, tahmin ile gerçek puanlar arasındaki farkların karelerinin ortalamasının kareköküdür.
        - Düşük RMSE değeri, modelin yüksek doğrulukla tahmin yaptığı anlamına gelir.
    """

    # L2Loss fonksiyonu, RMSE hesaplamak için uygun bir kayıp fonksiyonudur.
    loss = gluon.loss.L2Loss()

    # Toplam kaybı ve örnek sayısını biriktirmek için bir yardımcı nesne (Accumulator) oluşturuyoruz.
    # İlk parametre toplam kaybı, ikinci parametre ise toplam örnek sayısını biriktirir.
    metric = d2l.Accumulator(2)

    for i, (customers, movies, rates) in enumerate(data_iter):
        # Veri elemanlarını belirtilen cihazda (CPU/GPU) kullanabilmek için uygun şekilde taşıyoruz.
        customers, movies, rates = customers.as_in_ctx(ctx), movies.as_in_ctx(ctx), rates.as_in_ctx(ctx)

        # Modeli kullanarak tahminleri al.
        preds = net(customers, movies)

        # Tahminler ile gerçek puanlar arasındaki kaybı hesapla.
        l = loss(preds, rates)

        # Toplam kaybı ve örnek sayısını biriktir.
        # l.sum() toplam kaybı verir, rates.size ise batch'teki toplam örnek sayısını belirtir.
        metric.add(l.sum().item(), rates.size)

    # Toplam kaybı örnek sayısına bölüp, karekökünü alarak RMSE'yi hesaplıyoruz.
    # Bu işlem, tüm veri üzerinde ortalama hata miktarını ölçer.
    rmse = nd.sqrt(nd.array(metric[0]) / nd.array(metric[1]))

    return rmse.asscalar()

# Test veri kümesi üzerinde RMSE hesapla.
# Test verileri üzerinde modelin başarımını değerlendireceğiz.
test_rmse = evaluate_rmse(net, test_iter, ctx)

print(f'Test RMSE: {test_rmse:.4f}')

Test RMSE: 0.7567


## **Model Performans Değerlendirmesi: Film Önerisi Fonksiyonu**


In [10]:
import numpy as np
import random
import mxnet as mx

def recommend_movies(net, num_customers, num_movies, ctx, num_recommendations=10):
    """
    Rastgele bir customer seçer ve ona en yüksek tahmin edilen movie önerilerini sunar.

    Args:
        net (gluon.Block): Eğitimli öneri sistemi modeli, customer ve movie verilerini alarak tahmin yapar.
        num_customers (int): Sistemdeki toplam customer sayısı.
        num_movies (int): Sistemdeki toplam movie sayısı.
        ctx (mxnet.Context): Hesaplamaların yapılacağı cihaz (CPU veya GPU).
        num_recommendations (int): Customer'a önerilecek movie sayısı (varsayılan 10).

    Returns:
        list: Customer için önerilen MovieID'lerini içeren liste.
    """

    # Rastgele bir customer seç.
    customer_id = random.randint(0, num_customers - 1)

    # Customer'ın tüm movie'ler için tahminlerini hesapla.
    movie_ids = np.arange(num_movies)
    customers = np.full_like(movie_ids, customer_id)  # Tüm movie'ler için aynı customer.
    movies = movie_ids

    # Verileri cihazda (GPU veya CPU) işlenebilir hale getir.
    # mx.np.array kullanarak MXNet NumPy ndarray'leri oluşturuyoruz.
    customers = mx.np.array(customers, ctx=ctx)
    movies = mx.np.array(movies, ctx=ctx)

    # Modeli kullanarak tahminleri al.
    preds = net(customers, movies)

    # Tahmin edilen puanlar ile MovieID'lerini birleştir.
    pred_with_movie_id = list(zip(preds.asnumpy(), movie_ids))

    # Rate'lere göre azalan sırayla sıralanmış en iyi movie'leri öner.
    pred_with_movie_id.sort(reverse=True, key=lambda x: x[0])

    # En yüksek rate'li 10 movie'yi al.
    recommended_movies = [movie_id for _, movie_id in pred_with_movie_id[:num_recommendations]]

    return customer_id, recommended_movies

# 10 adet öneri al.
customer_id, recommended_movies = recommend_movies(net, num_customers, num_movies, ctx, 10)

print(f"Recommended movies for customer {customer_id}: {recommended_movies}")

Recommended movies for customer 19118: [9300, 3997, 5398, 8166, 6543, 469, 7627, 5976, 1330, 7382]
